# Etapa 1.1 — Fairness e Auditoria de Viés

**Disciplina:** Ética e Fairness em ML
**Foco:** Auditoria de viés do modelo campeão de churn em relação ao atributo sensível `gender`

Este notebook complementa o `eda_churn_prediction.ipynb` (Etapa 1), reaproveitando os dados
processados, os splits de treino/teste e os artefatos (`scaler.pkl`, `feature_columns.pkl`)
gerados naquela etapa, além do modelo campeão já treinado e avaliado (via MLflow) na
Etapa 2 (`modelagem_avaliacao_churn_prediction.ipynb`), para auditar seu comportamento em
relação ao atributo sensível `gender` e avaliar a necessidade de mitigação de viés.

**Tarefas desta etapa:**
1. Carregar os dados processados, splits e artefatos (scaler, colunas de features) da Etapa 1
2. Carregar o modelo campeão registrado no MLflow e gerar predições no conjunto de teste
3. Calcular métricas de fairness por subgrupo de gênero (`Female`/`Male`) com `fairlearn`: acurácia, selection rate, FPR e FNR (`MetricFrame`)
4. Analisar a diferença máxima entre grupos em cada métrica
5. Concluir sobre a necessidade (ou não) de aplicar técnicas de mitigação de viés

**Métrica técnica prioritária:** Recall (classe Churn) e ROC-AUC, dado o desbalanceamento do dataset (73,5%/26,5%).
**Métrica de negócio:** minimizar falsos negativos — o custo de não identificar um cliente propenso a cancelar supera o custo de uma ação de retenção desnecessária.
**Atributo sensível auditado:** `gender` (Female/Male).
**Métricas de fairness:** accuracy, selection rate, false positive rate (FPR) e false negative rate (FNR) por grupo, comparadas pela diferença máxima entre grupos.

**Resultado:** diferenças inferiores a 2 pontos percentuais entre os grupos em todas as métricas avaliadas, indicando ausência de viés relevante relacionado a gênero — consistente com a associação quase nula (Cramér's V = 0,009) identificada na EDA. Não foi necessário aplicar mitigação de viés.

**Entregável:** Notebook de auditoria de fairness documentado, com conclusão sobre a ausência de viés significativo por gênero.

## 1. Importar os dados pré-processados

In [20]:
import joblib
import pandas as pd
import os

print("=== CARREGAMENTO DOS DADOS E ARTEFATOS PRÉ-PROCESSADOS ===\n")

input_dir = "data/processed"

# 1. Carregar o dataframe processado completo (opcional, útil para referência/análises gerais)
df_encoded = pd.read_csv(os.path.join(input_dir, "telco_churn_processed.csv"))
print(f"✓ Dataframe processado carregado: {df_encoded.shape}")

# 2. Carregar os splits de treino e teste (mesmos usados no treino do modelo)
X_train = pd.read_csv(os.path.join(input_dir, "X_train.csv"))
X_test = pd.read_csv(os.path.join(input_dir, "X_test.csv"))
y_train = pd.read_csv(os.path.join(input_dir, "y_train.csv")).squeeze("columns")
y_test = pd.read_csv(os.path.join(input_dir, "y_test.csv")).squeeze("columns")
print(f"✓ Splits carregados — X_train: {X_train.shape}, X_test: {X_test.shape}")

# 3. Carregar o scaler ajustado no treino
scaler = joblib.load(os.path.join(input_dir, "scaler.pkl"))
print("✓ Scaler carregado")

# 4. Carregar a lista de colunas do encoding
feature_columns = joblib.load(os.path.join(input_dir, "feature_columns.pkl"))
print(f"✓ Lista de colunas carregada ({len(feature_columns)} features)")

print("\n✓ Carregamento concluído!")

=== CARREGAMENTO DOS DADOS E ARTEFATOS PRÉ-PROCESSADOS ===

✓ Dataframe processado carregado: (7043, 48)
✓ Splits carregados — X_train: (5634, 18), X_test: (1409, 18)
✓ Scaler carregado
✓ Lista de colunas carregada (18 features)

✓ Carregamento concluído!


In [16]:
import mlflow

model_uri = "runs:/f8da89fbaa0148d0afd4b254c272e515/model"
model = mlflow.sklearn.load_model(model_uri)

In [ ]:
from sklearn.model_selection import train_test_split
from fairlearn.metrics import MetricFrame, selection_rate, false_positive_rate, false_negative_rate
from sklearn.metrics import accuracy_score

# Reproduzir o mesmo split do treinamento para comparar os grupos na mesma amostra
# Reproduz o mesmo split usado no treino (mesmos parâmetros = mesmos índices),
# garantindo acesso a TODAS as colunas do df_encoded, incluindo o atributo sensível
X_full = df_encoded.drop(columns=['target'])
y_full = df_encoded['target']

X_train_full, X_test_full, y_train_full, y_test_full = train_test_split(
    X_full, y_full, test_size=0.2, random_state=42, stratify=y_full
)

# Gerar predições com o modelo carregado do MLflow
feature_columns = list(model.feature_names_in_)
X_test_model = X_test_full[feature_columns]

if hasattr(model, "named_steps"):
    # Pipelines já aplicam internamente o pré-processamento necessário
    y_pred = model.predict(X_test_model)
else:
    # Modelos sem pipeline precisam receber os dados transformados pelo scaler salvo
    X_test_scaled = scaler.transform(X_test_model)
    y_pred = model.predict(X_test_scaled)

# Atributo sensível: gender (após one-hot, restou apenas gender_Male: 0=Female, 1=Male)
sensitive_feature = X_test_full['gender_Male'].map({0: 'Female', 1: 'Male'})

# Calcular as métricas separadamente para cada grupo de gênero
# Criar MetricFrame para avaliar fairness
metric_frame = MetricFrame(
    metrics={
        'accuracy': accuracy_score,
        'selection_rate': selection_rate,
        'false_positive_rate': false_positive_rate,
        'false_negative_rate': false_negative_rate
    },
    y_true=y_test_full,
    y_pred=y_pred,
    sensitive_features=sensitive_feature
)

print("Métricas por grupo (gênero):")
print(metric_frame.by_group)
print("\nDiferença máxima entre grupos:")
print(metric_frame.difference())

Métricas por grupo (gênero):
             accuracy  selection_rate  false_positive_rate  \
gender_Male                                                  
Female       0.662300        0.548763             0.421053   
Male         0.660665        0.531856             0.414048   

             false_negative_rate  
gender_Male                       
Female                  0.124352  
Male                    0.116022  

Diferença máxima entre grupos:
accuracy               0.001635
selection_rate         0.016907
false_positive_rate    0.007005
false_negative_rate    0.008330
dtype: float64


### Conclusão da Análise de Fairness

O modelo apresentou diferenças inferiores a 2 pontos percentuais entre os grupos `Female` e `Male` em todas as métricas avaliadas (acurácia, taxa de seleção, FPR e FNR), indicando ausência de viés relevante relacionado ao gênero. Esse resultado é consistente com a análise exploratória, que já havia identificado associação estatística quase nula entre `gender` e a variável de churn (Cramér's V = 0,009). Não há evidência que justifique a aplicação de técnicas de mitigação de viés nesse atributo.